# 08b — Retropolación 2003-2015: análisis de pertinencia (Fase 1, D25 sección 5)

**Solo Fase 1**: construye el backcast de CBA/CBT (FACPCE, ancla abril-2016,
ventana `"a_mismo_trimestre"` confirmada en `08_linea_pobreza.ipynb`/D26),
calcula la tasa de pobreza/indigencia propia para 2003-2015 con ese backcast, y
la compara contra dos referencias externas -- CEDLAS/Tornarolli (2018, nivel +
banda de tolerancia, los 3 tramos) y UCA-EDSA (brecha de desvío estándar en el
puente 2016-2023, solo 2010-2015) -- para clasificar cada tramo
(2003-2006/2007-2009/2010-2015) como pertinente / pertinente con reservas / no
pertinente. **No llena `linea_pobreza`/`ratio_lp` en la era histórica todavía**
-- eso es la Fase 2, condicionada a esta clasificación.

**Resultado (ver diagnóstico al final de la notebook, D28)**: el backcast
FACPCE-ancla-2016 sobreestima la pobreza histórica muy por encima de las 3
referencias externas, de forma sistemática en todo el período -- **los 3
tramos dan `no_pertinente`**. La Fase 2 no arranca esta ronda.

**Lee** `data/02_nucleo/{hogar,individual}`, `data/01_armonizado/hogar` (para
`REGION`), `data/canastas_indec/` (canasta real + ancla), `data/indice-FACPCE.csv`,
`data/02b_pobreza/hogar/` (ya construida, para el puente 2016-2023 real) y
`data/canastas_indec/retropolacion/` (las 3 fuentes externas, D27).
**Escribe** `data/meta/retropolacion_tasas_propias_historicas.csv`,
`retropolacion_comparacion_{cedlas,indec_anterior}.csv`,
`retropolacion_pertinencia_{cedlas,uca}.csv`, `retropolacion_uca_detalle_z.csv`.

In [ ]:
import sys
import time
from pathlib import Path


def _raiz_repo(desde: Path | None = None) -> Path:
    actual = (desde or Path.cwd()).resolve()
    for candidato in (actual, *actual.parents):
        if (candidato / "pytest.ini").exists():
            return candidato
    raise FileNotFoundError(f"no se encontró la raíz del repo subiendo desde {actual}")


REPO = _raiz_repo()
sys.path.insert(0, str(REPO / "src"))

import pandas as pd

import pipeline_meta as pm
from armonizacion import es_historico, trimestres_publicados
import canasta_regional
import linea_pobreza
import retropolacion_pobreza

NUCLEO = REPO / "data" / "02_nucleo"
ARMONIZADO = REPO / "data" / "01_armonizado"
META = REPO / "data" / "meta"
CANASTAS = REPO / "data" / "canastas_indec"
POBREZA = REPO / "data" / "02b_distancia_lp" / "hogar"
RETROPOLACION = CANASTAS / "retropolacion"
FACPCE_CSV = REPO / "data" / "indice-FACPCE.csv"

METODO_FINAL = "a_mismo_trimestre"
TODOS_LOS_TRIMESTRES = trimestres_publicados()
TRIMESTRES_HISTORICOS = [t for t in TODOS_LOS_TRIMESTRES if t < (2016, 2)]
print(f"{len(TRIMESTRES_HISTORICOS)} trimestres históricos disponibles "
      f"({TRIMESTRES_HISTORICOS[0]}-{TRIMESTRES_HISTORICOS[-1]})")


## Fuentes externas (D27, ya descargadas/transcriptas)

In [ ]:
cedlas = pd.read_csv(RETROPOLACION / "cedlas" / "series_comparables_anexo_b.csv")
cedlas = cedlas.rename(columns={c: c.replace("indig_", "indigencia_").replace("pob_", "pobreza_")
                                 for c in cedlas.columns})
uca = pd.read_csv(RETROPOLACION / "uca" / "edsa_pobreza_indigencia_anual.csv")
indec_anterior = pd.read_csv(RETROPOLACION / "indec_metodologia_anterior" / "pobreza_1s2003_2s2012.csv")
print(f"CEDLAS: {len(cedlas)} semestres ({cedlas['semestre'].iloc[0]}-{cedlas['semestre'].iloc[-1]})")
print(f"UCA-EDSA: {len(uca)} años ({uca['ANO4'].min()}-{uca['ANO4'].max()})")
print(f"INDEC metodología anterior: {len(indec_anterior)} semestres ({indec_anterior['periodo'].iloc[0]}-{indec_anterior['periodo'].iloc[-1]})")


## Backcast de CBA/CBT (FACPCE, ancla abril-2016, ventana (a))

Verificación de borde: el backcast en el último mes (marzo 2016) debe coincidir
con la canasta real de abril 2016 escalada por la razón FACPCE mar/abr-2016 --
no exactamente 1 (son meses distintos), pero del orden de magnitud correcto.

In [ ]:
INICIO_BACKCAST = (2003, 1)
FIN_BACKCAST = (2016, 3)

indice_facpce = retropolacion_pobreza.cargar_facpce(FACPCE_CSV)
canasta_backcast = retropolacion_pobreza.construir_canasta_backcast_trimestral(
    CANASTAS, indice_facpce, METODO_FINAL, INICIO_BACKCAST, FIN_BACKCAST)
print(f"{len(canasta_backcast)} filas región×trimestre, "
      f"{canasta_backcast['ANO4'].min()}T{canasta_backcast.loc[canasta_backcast['ANO4'].idxmin(),'TRIMESTRE']} a "
      f"{canasta_backcast['ANO4'].max()}T{canasta_backcast.loc[canasta_backcast['ANO4'].idxmax(),'TRIMESTRE']}")
print(f"ventanas parciales: {int(canasta_backcast['canasta_ventana_parcial'].sum())}")
display(canasta_backcast[(canasta_backcast['ANO4'] == 2003) & (canasta_backcast['TRIMESTRE'] == 3)])


## Tasa de pobreza/indigencia propia, 2003-2015 (backcast)

Mismo criterio de ponderación que D17 (Gini): `PONDERA` del hogar (constante
entre sus integrantes, confirmado empíricamente, `retropolacion_pobreza.peso_hogar_historico`)
para hogares, `PONDERA × n_miembros` para personas -- no existe `PONDIH` antes
de 2016. Semestre = unión de sus 2 trimestres (como CEDLAS/INDEC). **2007-II**
no tiene datos propios (falta 2007T3, D1/CLAUDE.md) -- se arma con el mismo
semestre móvil que usan CEDLAS e INDEC (4ºT2007+1ºT2008), confirmado contra las
dos fuentes externas, no inventado acá. **2015-II** no es computable (falta
2015T3-2016T1 completo) -- queda fuera de la comparación, no se fuerza con el
mismo truco porque no hay ningún trimestre de 2016 anterior al T2 disponible.

In [ ]:
def _agregar_region(hogar: pd.DataFrame, anio: int, trimestre: int) -> pd.DataFrame:
    p = ARMONIZADO / "hogar" / f"ANO4={anio}" / f"TRIMESTRE={trimestre}" / "data.parquet"
    extra = pd.read_parquet(p, columns=["CODUSU", "NRO_HOGAR", "REGION"])
    extra["REGION"] = pd.to_numeric(extra["REGION"], errors="coerce").astype("Int8")
    return hogar.merge(extra, on=["CODUSU", "NRO_HOGAR"], how="left")


COLUMNAS_HOGAR_HIST = ["CODUSU", "NRO_HOGAR", "ANO4", "TRIMESTRE", "AGLOMERADO", "ITF", "ingreso_no_declarado"]
COLUMNAS_IND_HIST = ["CODUSU", "NRO_HOGAR", "COMPONENTE", "CH04", "CH06", "PONDERA"]


def construir_pobreza_historica(anio: int, trimestre: int) -> pd.DataFrame | None:
    hogar = pm.cargar_hogar(anio, trimestre, COLUMNAS_HOGAR_HIST, NUCLEO)
    individual = pm.cargar_individual(anio, trimestre, COLUMNAS_IND_HIST, NUCLEO)
    if hogar is None or individual is None:
        return None
    hogar = _agregar_region(hogar, anio, trimestre)
    canasta_t = canasta_backcast[(canasta_backcast["ANO4"] == anio) & (canasta_backcast["TRIMESTRE"] == trimestre)]
    resultado = linea_pobreza.calcular_distancia_lp(hogar, individual, canasta_t[["REGION", "cba_ae", "cbt_ae"]])
    peso = retropolacion_pobreza.peso_hogar_historico(individual)
    n_miembros = individual.groupby(["CODUSU", "NRO_HOGAR"]).size().rename("n_miembros")
    idx = pd.MultiIndex.from_frame(resultado[["CODUSU", "NRO_HOGAR"]])
    resultado["PONDERA"] = peso.reindex(idx).to_numpy()
    resultado["n_miembros"] = n_miembros.reindex(idx).to_numpy()
    return resultado


def tasa_ponderada(df: pd.DataFrame, columna: str, peso: str) -> float:
    valido = df[columna].notna() & df[peso].notna()
    num = (df.loc[valido, columna].astype(float) * df.loc[valido, peso]).sum()
    den = df.loc[valido, peso].sum()
    return 100 * num / den


def tasas_semestre_historico(componentes: list[tuple[int, int]]) -> dict | None:
    partes = []
    for anio, trimestre in componentes:
        df = construir_pobreza_historica(anio, trimestre)
        if df is None:
            return None
        partes.append(df)
    df = pd.concat(partes, ignore_index=True)
    df["peso_personas"] = df["PONDERA"] * df["n_miembros"]
    return {
        "pobreza_hogares": tasa_ponderada(df, "pobre", "PONDERA"),
        "pobreza_personas": tasa_ponderada(df, "pobre", "peso_personas"),
        "indigencia_hogares": tasa_ponderada(df, "indigente", "PONDERA"),
        "indigencia_personas": tasa_ponderada(df, "indigente", "peso_personas"),
    }


In [ ]:
def semestres_historicos_computables(trimestres_disponibles: list[tuple[int, int]]):
    disponibles = set(trimestres_disponibles)
    resultado = []
    for anio in range(2003, 2016):
        for semestre, (t1, t2) in ((1, (1, 2)), (2, (3, 4))):
            if anio == 2003 and semestre == 1:
                continue  # la EPH continua arranca en 2003T3, no hay 1S2003
            componentes = ((anio, t1), (anio, t2))
            if componentes[0] in disponibles and componentes[1] in disponibles:
                resultado.append((anio, semestre, componentes, False))
            elif anio == 2007 and semestre == 2:
                movil = ((2007, 4), (2008, 1))
                if movil[0] in disponibles and movil[1] in disponibles:
                    resultado.append((anio, semestre, movil, True))
    return resultado


SEMESTRES_HIST = semestres_historicos_computables(TRIMESTRES_HISTORICOS)
print(f"{len(SEMESTRES_HIST)} semestres históricos computables con datos propios")
faltantes = [(a, s) for a in range(2003, 2016) for s in (1, 2)
             if (a, s) != (2003, 1) and (a, s) not in {(x[0], x[1]) for x in SEMESTRES_HIST}]
print(f"semestres sin dato propio (huecos D1/D7): {faltantes}")


In [ ]:
t0_hist = time.time()
filas_propias = []
for anio, semestre, componentes, es_movil in SEMESTRES_HIST:
    tasas = tasas_semestre_historico(componentes)
    if tasas is None:
        continue
    filas_propias.append({"ANO4": anio, "SEMESTRE": semestre, "semestre_movil": es_movil, **tasas})
propia_historica = pd.DataFrame(filas_propias)
propia_historica.to_csv(META / "retropolacion_tasas_propias_historicas.csv", index=False)
print(f"{len(propia_historica)} semestres calculados, {time.time()-t0_hist:.1f}s")
display(propia_historica)


## Mecanismo CEDLAS: nivel + banda de tolerancia (D25, sección 5.c)

CEDLAS reporta **personas**, no hogares (confirmado en D25: su "Serie
Comparable" para 2016-II/2017-I coincide con la tasa oficial de personas que
ya validamos) -- la comparación es solo contra `pobreza_personas`/
`indigencia_personas`. Banda = rango entre `serie_comparable`, `est3` (hot-deck
propio) y `sin_correccion_nr` (exclusión) -- las 3 variantes de tratamiento de
no respuesta que trae el propio Anexo B.

In [ ]:
comparacion_cedlas = propia_historica.merge(cedlas, on=["ANO4", "SEMESTRE"], how="inner", suffixes=("", "_cedlas"))

for indicador in ("pobreza", "indigencia"):
    columnas_banda = [f"{indicador}_serie_comparable", f"{indicador}_est3", f"{indicador}_sin_correccion_nr"]
    comparacion_cedlas[f"{indicador}_dentro_banda"] = comparacion_cedlas.apply(
        lambda fila, cb=columnas_banda, ind=indicador: retropolacion_pobreza.clasificar_dentro_de_banda(
            fila[f"{ind}_personas"], [fila[c] for c in cb]), axis=1)

comparacion_cedlas.to_csv(META / "retropolacion_comparacion_cedlas.csv", index=False)
print(f"{len(comparacion_cedlas)} semestres comparados contra CEDLAS "
      f"(de {len(propia_historica)} propios -- CEDLAS llega hasta 2017-I, cubre todos los históricos)")
display(comparacion_cedlas[["ANO4", "SEMESTRE", "semestre_movil", "pobreza_personas", "pobreza_serie_comparable",
                             "pobreza_dentro_banda", "indigencia_personas", "indigencia_serie_comparable",
                             "indigencia_dentro_banda"]])


In [ ]:
TRAMOS = {"2003-2006": (2003, 2006), "2007-2009": (2007, 2009), "2010-2015": (2010, 2015)}

filas_pertinencia_cedlas = []
for tramo, (anio_ini, anio_fin) in TRAMOS.items():
    sub = comparacion_cedlas[(comparacion_cedlas["ANO4"] >= anio_ini) & (comparacion_cedlas["ANO4"] <= anio_fin)]
    for indicador in ("pobreza", "indigencia"):
        fuera = (~sub[f"{indicador}_dentro_banda"]).tolist()
        filas_pertinencia_cedlas.append({
            "tramo": tramo, "indicador": indicador, "n_semestres": len(sub),
            "n_fuera_de_banda": sum(fuera),
            "pertinencia": retropolacion_pobreza.clasificar_tramo_cedlas(fuera),
        })

pertinencia_cedlas = pd.DataFrame(filas_pertinencia_cedlas)
pertinencia_cedlas.to_csv(META / "retropolacion_pertinencia_cedlas.csv", index=False)
display(pertinencia_cedlas)


## Brecha propia − CEDLAS por semestre (pedido de Iván tras revisar D28)

El diagnóstico de D28 ("salto de nivel por cambio de metodología de canasta")
predice una brecha aproximadamente **constante** en el tiempo. Si en cambio la
brecha se achica a medida que el semestre se acerca al ancla (abril 2016), es
la firma de un **índice que acumula menos inflación que la real** entre ese
semestre y el ancla -- el backcast multiplica la canasta ancla por un factor
`FACPCE(t)/FACPCE(ancla)` demasiado alto cuanto más atrás en el tiempo, porque
FACPCE subestima la inflación acumulada hasta 2016, no porque la canasta 2016
sea "otra cosa".

In [ ]:
brecha_cedlas = comparacion_cedlas[["ANO4", "SEMESTRE", "semestre_movil"]].copy()
brecha_cedlas["brecha_pobreza_pp"] = comparacion_cedlas["pobreza_personas"] - comparacion_cedlas["pobreza_serie_comparable"]
brecha_cedlas["brecha_indigencia_pp"] = comparacion_cedlas["indigencia_personas"] - comparacion_cedlas["indigencia_serie_comparable"]
brecha_cedlas.to_csv(META / "retropolacion_brecha_cedlas_por_semestre.csv", index=False)
display(brecha_cedlas)


In [ ]:
import matplotlib.pyplot as plt

etiquetas = comparacion_cedlas["ANO4"].astype(str) + "-" + comparacion_cedlas["SEMESTRE"].map({1: "I", 2: "II"})
x = range(len(etiquetas))

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(x, brecha_cedlas["brecha_pobreza_pp"], marker="o", label="pobreza")
ax.plot(x, brecha_cedlas["brecha_indigencia_pp"], marker="o", label="indigencia")
ax.axhline(0, color="grey", linewidth=0.8)
ax.set_xticks(list(x))
ax.set_xticklabels(etiquetas, rotation=90)
ax.set_ylabel("brecha propia − CEDLAS, personas (pp)")
ax.set_title("Brecha propia vs. CEDLAS (Serie Comparable) -- se achica monótonamente hacia\n"
              "el ancla (abril 2016, justo después del último punto, 2015-I)")
ax.legend()
plt.tight_layout()
plt.show()

print(f"brecha pobreza: {brecha_cedlas['brecha_pobreza_pp'].iloc[0]:.1f}pp (2003-II) -> "
      f"{brecha_cedlas['brecha_pobreza_pp'].iloc[-1]:.1f}pp (2015-I)")
print(f"brecha indigencia: {brecha_cedlas['brecha_indigencia_pp'].iloc[0]:.1f}pp (2003-II) -> "
      f"{brecha_cedlas['brecha_indigencia_pp'].iloc[-1]:.1f}pp (2015-I)")


## Composición de FACPCE -- qué índice usa en cada tramo

**[confirmado, fuente primaria, leída directamente -- no de un resumen
automático]** FACPCE, *Resolución Técnica N°6, Estados Contables en Moneda
Homogénea. Índice a utilizar de acuerdo con la Res. JG N°539/18*
(`https://www.facpce.org.ar/wp-content/uploads/2020/05/MetodologiaIndices.pdf`):

> *"a) Desde enero de 1993 hasta octubre de 2015: se incluyó el índice **IPIM**
> del INDEC... b) Desde noviembre 2015 hasta diciembre 2016: se incluyó el
> **IPIM calculado por FACPCE**... c) Desde enero 2017 en adelante: se
> utilizará el **IPC nacional** publicado por el INDEC."*

**Todo el período que nos interesa (2003-2016) usa IPIM, no IPC** -- el
Índice de Precios Internos al por Mayor (precios mayoristas/productor), no un
índice de precios al consumidor. Confirmado también en el propio archivo:
`data/indice-FACPCE.xlsx` trae la columna titulada literalmente
`"IPC NACIONAL EMPALME IPIM"` (`xl/sharedStrings.xml`, nunca auditado antes --
D25/D26 solo auditaron el CSV). Un índice mayorista y uno de consumo minorista
son canastas distintas (el IPIM no incluye servicios, alquileres, tarifas
reguladas) y pueden divergir mucho en una ventana de 13 años, sobre todo en un
período con tarifas de servicios congeladas (2007-2015) -- es la explicación
estructural más directa de por qué un empalme con IPIM atrasa la inflación
acumulada frente a un IPC real.

## Factor de inflación implícito en el backcast, y con qué contrastarlo

`CBT_ancla(abril-2016, GBA) / CBT_backcast(2003T3, GBA)` -- el multiplicador
de precios que el backcast le atribuye a los ~12,6 años entre el 3er
trimestre de 2003 y abril de 2016.

In [ ]:
cbt_largo = canasta_regional.cargar_canasta_larga(CANASTAS / "cbt_regional_crudo.csv", "cbt_ae")
cbt_ancla_gba = cbt_largo[(cbt_largo["ANO4"] == 2016) & (cbt_largo["MES"] == 4) & (cbt_largo["REGION"] == 1)]["cbt_ae"].iloc[0]
cbt_backcast_2003t3_gba = canasta_backcast[(canasta_backcast["ANO4"] == 2003) & (canasta_backcast["TRIMESTRE"] == 3)
                                            & (canasta_backcast["REGION"] == 1)]["cbt_ae"].iloc[0]
factor_implicito = cbt_ancla_gba / cbt_backcast_2003t3_gba
print(f"CBT ancla (abril 2016, GBA): $ {cbt_ancla_gba:,.2f}")
print(f"CBT backcast (2003T3, GBA): $ {cbt_backcast_2003t3_gba:,.2f}")
print(f"factor implícito FACPCE, sep-2003 -> abr-2016: {factor_implicito:.3f}x "
      f"({100*(factor_implicito-1):.1f}% de inflación acumulada implícita)")


**No se puede extraer un factor equivalente de CEDLAS**: el Anexo B trae solo
tasas de incidencia (5 variantes), no el valor en pesos de su CBA/CBT -- para
sacar su factor implícito habría que rehacer su cálculo completo, fuera de
alcance de este chequeo puntual.

**Lo que sí hay, documentado, sin bajarlo todavía (pedido de Iván)**:

1. **El propio empalme de Tornarolli/CEDLAS** (ya citado en D25, Anexo A de su
   paper, ya en el repo): *"IPC-INDEC período 2003-2006, **IPC-Graciela
   Bevacqua período 2007-2015**, e IPC-INDEC período 2016-2017"* -- Bevacqua
   fue la responsable técnica del IPC-INDEC desplazada en 2007, y su índice
   independiente es la referencia académica más citada para ese tramo
   precisamente porque el IPC oficial (y, por lo visto acá, también el IPIM
   que linkea FACPCE) no era confiable en ese período. **Es la alternativa
   más directa a evaluar**: ya tenemos el paper que la documenta, no hay que
   salir a buscar una fuente nueva.
2. **INDEC, "Valorización mensual de la Canasta Básica de Alimentos (CBA) y
   de la Canasta Básica Total (CBT), Aglomerado Gran Buenos Aires"**
   (`http://www.indec.gov.ar/principal.asp?id_tema=536`, citado en el pie de
   página de `pob_tot_2sem12.pdf`, ya en el repo) -- si todavía está vigente,
   daría el valor en pesos de la CBA/CBT **metodología anterior, real, sin
   deflactar nada** para gran parte de 2003-2015, el contraste más directo
   posible contra el backcast FACPCE.

**Propuesta (no implementada, a confirmar)**: si se decide mejorar el
backcast, evaluar primero la opción 2 (valores reales, no requiere deflactar)
y, si no está disponible para todo el tramo, la opción 1 (empalme de
Tornarolli, ya documentado y parcialmente en el repo).

## Chequeo de bug (para descartar, pedido de Iván)

### a. Cuenta a mano de un hogar real de 2003-II

Hogar `CODUSU=125303, NRO_HOGAR=1`, 2003T3, región GBA, `ITF=3300`. Integrantes:
varón 33 años (1,00, banda 30-45), mujer 32 años (0,77, banda 30-45), mujer de
2 años (0,46, Tabla 1) -- `unidades_ae = 1,00 + 0,77 + 0,46 = 2,23`.

`cbt_ae` backcast 2003T3 GBA = 670,93 -> `linea_pobreza = 670,93 × 2,23 =
1.496,18` -> `ratio_lp = 3300 / 1.496,18 = 2,206`. **Coincide exactamente con
lo que calcula el código** (celda de abajo) -- la fórmula no tiene un error de
cálculo, el problema es el nivel de `cbt_ae`, no cómo se usa.

In [ ]:
hogar_2003 = pm.cargar_hogar(2003, 3, ["CODUSU", "NRO_HOGAR", "ANO4", "TRIMESTRE", "AGLOMERADO", "ITF", "ingreso_no_declarado"], NUCLEO)
individual_2003 = pm.cargar_individual(2003, 3, ["CODUSU", "NRO_HOGAR", "COMPONENTE", "CH04", "CH06", "PONDERA"], NUCLEO)
hogar_2003 = _agregar_region(hogar_2003, 2003, 3)
canasta_2003t3 = canasta_backcast[(canasta_backcast["ANO4"] == 2003) & (canasta_backcast["TRIMESTRE"] == 3)]
resultado_2003 = linea_pobreza.calcular_distancia_lp(hogar_2003, individual_2003, canasta_2003t3[["REGION", "cba_ae", "cbt_ae"]])
display(resultado_2003[(resultado_2003["CODUSU"] == "125303") & (resultado_2003["NRO_HOGAR"] == 1)].T)


### b. Escala de `ITF` 2003-2015 vs. 2016+ -- descarta un problema de unidades

Si `ITF` estuviera en una unidad distinta en algún tramo (ej. un factor
10x/1000x de diferencia), la mediana nominal tendría un salto discontinuo. No
lo tiene -- crece suave y monótona en todo el período, consistente con la
inflación nominal conocida de Argentina, sin ningún quiebre de escala:

In [ ]:
filas_escala = []
for anio, trimestre in [(2003, 3), (2008, 2), (2013, 4), (2016, 2), (2020, 2), (2024, 3)]:
    h = pm.cargar_hogar(anio, trimestre, ["ITF"], NUCLEO)
    itf_positivo = h["ITF"].dropna()
    itf_positivo = itf_positivo[itf_positivo > 0]
    filas_escala.append({"ANO4": anio, "TRIMESTRE": trimestre, "n": len(itf_positivo),
                          "mediana_itf": itf_positivo.median(), "p90_itf": itf_positivo.quantile(0.9)})
display(pd.DataFrame(filas_escala))


## Mecanismo UCA: brecha de desvío estándar en el puente 2016-2023 (D25, 5.c)

Solo aplica al tramo **2010-2015** (único donde EDSA Bicentenario es
relevamiento propio, no un backcast, D25/D27). El puente usa la tasa **propia
real** (no backcast) para 2016-2023, ya construida en `08_linea_pobreza.ipynb`
(`data/02b_pobreza/hogar/`, método (a) confirmado) -- sin pasar por el
backcast, que ahí no hace falta.

In [ ]:
def tasas_semestre_real(anio: int, semestre: int) -> dict | None:
    trimestres = (1, 2) if semestre == 1 else (3, 4)
    partes = []
    for t in trimestres:
        p = POBREZA / f"ANO4={anio}" / f"TRIMESTRE={t}" / "data.parquet"
        if not p.exists():
            return None
        df = pd.read_parquet(p)
        pind = NUCLEO / "individual" / f"ANO4={anio}" / f"TRIMESTRE={t}" / "data.parquet"
        n_miembros = (pd.read_parquet(pind, columns=["CODUSU", "NRO_HOGAR"])
                      .groupby(["CODUSU", "NRO_HOGAR"]).size().rename("n_miembros"))
        df = df.merge(n_miembros, on=["CODUSU", "NRO_HOGAR"], how="left")
        partes.append(df)
    df = pd.concat(partes, ignore_index=True)
    df["peso_personas"] = df["PONDIH"] * df["n_miembros"]
    return {
        "pobreza_personas": tasa_ponderada(df, "pobre", "peso_personas"),
        "indigencia_personas": tasa_ponderada(df, "indigente", "peso_personas"),
    }


ANIOS_PUENTE = list(range(2016, 2024))
propia_2s_puente = {anio: tasas_semestre_real(anio, 2) for anio in ANIOS_PUENTE}
print({a: {k: round(v, 1) for k, v in r.items()} for a, r in propia_2s_puente.items()})


In [ ]:
uca_por_anio = uca.set_index("ANO4")
propia_2s_tramo_uca = {
    fila["ANO4"]: {"pobreza_personas": fila["pobreza_personas"], "indigencia_personas": fila["indigencia_personas"]}
    for _, fila in propia_historica[(propia_historica["SEMESTRE"] == 2)
                                     & (propia_historica["ANO4"].between(2010, 2015))].iterrows()
}
print(f"años 2010-2015 con dato propio (2S): {sorted(propia_2s_tramo_uca)}")

filas_pertinencia_uca = []
detalle_z_uca = []
for indicador, columna_uca in (("pobreza", "pobreza_edsa_personas"), ("indigencia", "indigencia_edsa_personas")):
    serie_propia_puente = {a: propia_2s_puente[a][f"{indicador}_personas"] for a in ANIOS_PUENTE}
    serie_uca_puente = uca_por_anio.loc[ANIOS_PUENTE, columna_uca].to_dict()
    serie_propia_tramo = {a: propia_2s_tramo_uca[a][f"{indicador}_personas"] for a in propia_2s_tramo_uca}
    serie_uca_tramo = uca_por_anio.loc[list(propia_2s_tramo_uca), columna_uca].to_dict()

    for excluir_2020 in (False, True):
        puente = retropolacion_pobreza.brecha_puente_uca(serie_propia_puente, serie_uca_puente,
                                                           ANIOS_PUENTE, excluir_2020=excluir_2020)
        z_por_anio = {a: (serie_propia_tramo[a] - serie_uca_tramo[a] - puente["media"]) / puente["std"]
                      for a in serie_propia_tramo}
        pertinencia = retropolacion_pobreza.clasificar_tramo_uca(z_por_anio)
        filas_pertinencia_uca.append({
            "indicador": indicador, "excluye_2020_del_puente": excluir_2020,
            "media_brecha_puente": round(puente["media"], 2), "std_brecha_puente": round(puente["std"], 2),
            "n_anios_fuera_2std": sum(abs(z) > 2 for z in z_por_anio.values()),
            "pertinencia_2010_2015": pertinencia,
        })
        for anio, z in z_por_anio.items():
            detalle_z_uca.append({"indicador": indicador, "excluye_2020_del_puente": excluir_2020,
                                   "ANO4": anio, "z": round(z, 2)})

pertinencia_uca = pd.DataFrame(filas_pertinencia_uca)
pertinencia_uca.to_csv(META / "retropolacion_pertinencia_uca.csv", index=False)
pd.DataFrame(detalle_z_uca).to_csv(META / "retropolacion_uca_detalle_z.csv", index=False)
display(pertinencia_uca)


## INDEC metodología anterior -- triangulación de contexto (2003-2006), sin criterio de pertinencia propio

D25 sección 5.a: esta fuente no decide nada por sí sola (es la misma fuente
oficial bajo reserva metodológica a partir de 2007, D3/CLAUDE.md) -- se reporta
solo como tercera referencia de contexto para el tramo 2003-2006, donde sí
predata la intervención.

In [ ]:
comparacion_indec_anterior = propia_historica[propia_historica["ANO4"] <= 2006].merge(
    indec_anterior[["ANO4", "SEMESTRE", "pobreza_hogares", "pobreza_personas",
                     "indigencia_hogares", "indigencia_personas"]],
    on=["ANO4", "SEMESTRE"], how="inner", suffixes=("_propia", "_indec_anterior"))
comparacion_indec_anterior.to_csv(META / "retropolacion_comparacion_indec_anterior.csv", index=False)
display(comparacion_indec_anterior[["ANO4", "SEMESTRE", "pobreza_personas_propia", "pobreza_personas_indec_anterior",
                                     "indigencia_personas_propia", "indigencia_personas_indec_anterior"]])


## Resumen de pertinencia por tramo, y desacuerdo CEDLAS vs. UCA (2010-2015)

No se elige una sola fuente cuando difieren (D25, 5.c) -- se reportan las dos
clasificaciones de 2010-2015 lado a lado.

In [ ]:
print("=== CEDLAS (nivel + banda de tolerancia, pobreza/indigencia personas) ===")
display(pertinencia_cedlas)

print()
print("=== UCA (brecha de desvío estándar en el puente, solo 2010-2015) ===")
display(pertinencia_uca)

tramo_2010_2015_cedlas = set(pertinencia_cedlas.loc[pertinencia_cedlas["tramo"] == "2010-2015", "pertinencia"])
tramo_2010_2015_uca = set(pertinencia_uca["pertinencia_2010_2015"])
if tramo_2010_2015_cedlas != tramo_2010_2015_uca:
    print(f"\nDESACUERDO 2010-2015: CEDLAS da {tramo_2010_2015_cedlas}, UCA da {tramo_2010_2015_uca} "
          f"-- no se elige una, queda para que Iván decida (D25, 5.c).")
else:
    print(f"\nCEDLAS y UCA coinciden en 2010-2015: {tramo_2010_2015_cedlas or tramo_2010_2015_uca}")


## Diagnóstico corregido (D29 -- revisa el diagnóstico original de D28)

**El diagnóstico de D28 ("salto de nivel por cambio de composición de
canasta, ENGHo 85/86 vs. 96/97") está refutado por la propia evidencia de
esta notebook** (D29, ver también el log): un cambio de composición
produciría una brecha aproximadamente constante en el tiempo, y lo que se ve
es una brecha que se achica de forma casi monótona a medida que el semestre
se acerca al ancla (abril 2016) -- la firma de un índice que **subestima la
inflación acumulada**, no de un salto de nivel puntual.

**Causa confirmada**: `data/indice-FACPCE.csv` empalma, para **todo
2003-2016**, el **IPIM** (índice de precios mayoristas/productor), no el IPC
(consumidor) -- confirmado contra la metodología oficial de FACPCE (Res. JG
539/18) y contra el propio archivo (`"IPC NACIONAL EMPALME IPIM"`). El factor
de inflación implícito que el backcast le atribuye a sep-2003→abr-2016 queda
calculado arriba; CEDLAS, en cambio, deflacta ese mismo tramo con el índice
de Graciela Bevacqua (su propio empalme, documentado en su Anexo A, ya
citado en D25) -- un IPC, no un IPIM, construido específicamente porque el
oficial (y, visto acá, también el mayorista que usa FACPCE) no era confiable
en el período intervenido.

**El chequeo de bug (arriba) descarta un error de cálculo o de unidades**: la
cuenta a mano de un hogar real coincide exactamente con el código, y la
escala nominal de `ITF` es suave y sin saltos en todo 2003-2024 -- el
problema está específicamente en el nivel de `cba_ae`/`cbt_ae` que produce el
backcast, no en cómo se usa.

**Conclusión de la Fase 1, sigue en pie**: con el backcast tal como está
construido (FACPCE/IPIM), **ningún tramo es pertinente** -- la clasificación
`no_pertinente` de arriba no cambia. Lo que cambia es el diagnóstico de *por
qué*, lo que sí importa para decidir el próximo paso: no es un límite
estructural del enfoque "deflactar la canasta 2016 hacia atrás" en general,
es específico de haber usado FACPCE/IPIM como deflactor -- un índice de
precios al consumidor mejor elegido (ej. el empalme de Bevacqua que usa
CEDLAS) podría dar un resultado muy distinto. Esta decisión de alcance queda
para que Iván la tome, no se implementa en esta ronda.

---

## Punto de parada: Fase 1 completa, concluye `no_pertinente`, Fase 2 no arranca

**No se llena `linea_pobreza`/`ratio_lp`/`linea_retropolada` en la era
histórica** -- la clasificación de arriba (CEDLAS por tramo, UCA para
2010-2015, INDEC metodología anterior como contexto) da `no_pertinente` en
los 3 tramos, sin desacuerdo entre CEDLAS y UCA. Mientras no se mejore el
backcast (ej. el método de valorización de Tornarolli, descrito pero no
implementado esta ronda, D25), la columna central de la etiqueta de pobreza
queda `NA` para 2003T3-2015T2, documentado -- esto es todo lo que corre esta
notebook.

---

# Ronda siguiente (pedido de Iván): tamaño de muestra por era + backcast Bevacqua

Dos partes, **solo análisis -- no se llena nada en `data/02b_pobreza/hogar`**.

## 1. Tamaño de muestra por era

`data/03_panel/pares_h4` ya trae los pares hogar `t → t+4` con el criterio de
panel del repo (`CODUSU+NRO_HOGAR`, D9) -- se usa tal cual, no se reconstruye
nada. `identidad_confirmada` es el mismo flag de calidad de vinculación que ya
usa `cascada.py` (D24).

In [ ]:
PANEL_DIR = REPO / "data" / "03_panel" / "pares_h4"

pares_h4 = pd.concat([pd.read_parquet(p) for p in sorted(PANEL_DIR.rglob("data.parquet"))], ignore_index=True)
confirmados_h4 = pares_h4[pares_h4["identidad_confirmada"] == True].copy()
confirmados_h4["era_t"] = confirmados_h4.apply(
    lambda r: "historica" if es_historico(r["ANO4_t"], r["TRIMESTRE_t"]) else "regular", axis=1)
confirmados_h4["era_th"] = confirmados_h4.apply(
    lambda r: "historica" if es_historico(r["ANO4_th"], r["TRIMESTRE_th"]) else "regular", axis=1)

print(f"pares_h4 totales (cualquier identidad): {len(pares_h4):,}")
print(f"identidad_confirmada: {int((pares_h4['identidad_confirmada'] == True).sum()):,}")
print()
print("hogares en t con match confirmado en t+4, por era de t:")
display(confirmados_h4["era_t"].value_counts())


### Pares que cruzan el corte metodológico (2015T2→2016T2 y cualquier otro historico→regular)

In [ ]:
cruces = int((confirmados_h4["era_t"] != confirmados_h4["era_th"]).sum())
print(f"pares con era_t != era_th en data/03_panel/pares_h4: {cruces}")

# Verificación directa, no solo inferida del resultado vacío de la capa ya construida.
hogar_2015t2 = pd.read_parquet(NUCLEO / "hogar" / "ANO4=2015" / "TRIMESTRE=2" / "data.parquet", columns=["CODUSU"])
hogar_2016t2 = pd.read_parquet(NUCLEO / "hogar" / "ANO4=2016" / "TRIMESTRE=2" / "data.parquet", columns=["CODUSU"])
print(f"CODUSU 2015T2, ejemplo: {hogar_2015t2['CODUSU'].iloc[0]!r}")
print(f"CODUSU 2016T2, ejemplo: {hogar_2016t2['CODUSU'].iloc[0]!r}")
print(f"CODUSU en común entre 2015T2 y 2016T2: {len(set(hogar_2015t2['CODUSU']) & set(hogar_2016t2['CODUSU']))}")


**0 pares cruzan la frontera, y no es una exclusión de política** -- es
estructural: `CODUSU` es un esquema de identificador completamente distinto
entre las dos eras (D7), numérico corto en la histórica
(ej. `393378`) vs. alfanumérico largo en la regular
(ej. `TQRMNOQUXHKOKMCDEGKDB00475140`). El join exacto de
`panel.vincular_hogares` no encuentra un solo `CODUSU` en común entre 2015T2
y 2016T2 -- confirmado acá directamente, no solo inferido del resultado vacío
de `data/03_panel/pares_h4`. Implicación para el alcance: la era histórica
nunca puede aportar un par que "cruce" hacia la etiqueta de pobreza 2016+ --
solo puede aportar pares **enteramente dentro** de la era histórica, con su
propia etiqueta (bloqueada por D28/D29 mientras el backcast no sea
pertinente).

### Hogares pobres en `t`, era regular (`ratio_lp` ya construido, `08_linea_pobreza.ipynb`/D26)

In [ ]:
pobreza_partes = [pd.read_parquet(p, columns=["CODUSU", "NRO_HOGAR", "ANO4", "TRIMESTRE", "pobre"])
                   for p in sorted(POBREZA.rglob("data.parquet"))]
pobreza_todos = pd.concat(pobreza_partes, ignore_index=True)

regular_h4 = confirmados_h4[confirmados_h4["era_t"] == "regular"].merge(
    pobreza_todos, left_on=["CODUSU", "NRO_HOGAR_t", "ANO4_t", "TRIMESTRE_t"],
    right_on=["CODUSU", "NRO_HOGAR", "ANO4", "TRIMESTRE"], how="left")

print(f"hogares regular con match confirmado en t+4: {len(regular_h4):,}")
print(f"con ratio_lp/pobre_t no nulo: {int(regular_h4['pobre'].notna().sum()):,} "
      f"({100 * regular_h4['pobre'].notna().mean():.1f}%, el resto es no-respuesta confirmada, D25 3.B)")
display(regular_h4["pobre"].value_counts(dropna=False))


### Corte con el holdout actual (2023T1+, D24)

In [ ]:
HOLDOUT_DESDE = (2023, 1)
confirmados_h4["es_holdout"] = confirmados_h4.apply(
    lambda r: (r["ANO4_t"], r["TRIMESTRE_t"]) >= HOLDOUT_DESDE, axis=1)

resumen_holdout = confirmados_h4.groupby(["era_t", "es_holdout"], as_index=False).size().rename(columns={"size": "n_pares"})
resumen_holdout.to_csv(META / "retropolacion_muestra_holdout.csv", index=False)
display(resumen_holdout)

n_train_solo_2016 = int(confirmados_h4[(confirmados_h4["era_t"] == "regular") & (~confirmados_h4["es_holdout"])].shape[0])
n_historico_total = int(confirmados_h4[confirmados_h4["era_t"] == "historica"].shape[0])
n_train_con_historico = n_train_solo_2016 + n_historico_total
print(f"entrenamiento, solo 2016+ (pre-2023T1): {n_train_solo_2016:,} pares")
print(f"entrenamiento, 2016+ más histórico: {n_train_con_historico:,} pares "
      f"({n_train_con_historico / n_train_solo_2016:.2f}x -- toda la era histórica es, por construcción, "
      f"anterior al holdout 2023T1)")


**Síntesis sección 1**: 206.350 hogares regulares con match confirmado en
`t+4` (166.475 con `ratio_lp` no nulo, 45.389 pobres en `t`) contra 243.901
hogares históricos con match confirmado -- la era histórica, si estuviera
disponible, **más que duplicaría** el universo de entrenamiento (160.786 →
404.687 pares, 2,52x) para el horizonte `h=4`. Es la cifra que hace que valga
la pena seguir evaluando el backcast en la sección 2, en vez de descartar la
era histórica sin más.

## 2. Backcast alternativo con el empalme de Bevacqua -- bloqueado en la fuente de datos

`src/retropolacion_pobreza.py` ya quedó parametrizado por índice (D30,
aditivo): `backcast_canasta_larga`/`construir_canasta_backcast_trimestral`
ahora reciben un `indice: pd.Series` ya cargado (antes recibían directamente
`facpce_csv` y lo cargaban adentro) -- cualquier índice con el shape
`cargar_indice_precios(ruta_csv, columna_valor)` sirve, no solo FACPCE.
`cargar_facpce` sigue andando igual que antes (wrapper fino, mismos tests).

In [ ]:
import inspect
print(inspect.signature(retropolacion_pobreza.construir_canasta_backcast_trimestral))
print(inspect.signature(retropolacion_pobreza.cargar_indice_precios))


### No se pudo conseguir la serie mensual de Bevacqua como dataset descargable

**Escalado, como pide CLAUDE.md ante una falla de acceso -- no se trabaja
alrededor sin avisar.** Búsqueda hecha esta ronda, sin éxito:

1. El propio PDF de Tornarolli/CEDLAS (`tornarolli_2018_documento_trabajo_226.pdf`,
   ya en el repo) menciona a Bevacqua **una sola vez**, en el Anexo A, sin cita a
   ninguna fuente de datos ni URL (`pdftotext` + `grep -i bevacqua`, todo el
   documento).
2. El CV de Tornarolli (CEDLAS) no trae enlace a datos de replicación del
   documento de trabajo 226.
3. Un artículo relacionado pero **distinto** (Gasparini et al., Redalyc,
   *"Pobreza e indigencia en Argentina: construcción de una serie completa y
   metodológicamente homogénea"*) sí cita una fuente concreta para 2007-2015 --
   pero usa **IPC-San Luis (desde 2012, + CABA y Córdoba)**, no Bevacqua, y el
   dato está en un Google Drive citado en su nota 5, no en este repo.
4. Búsquedas directas (IPC-Congreso, IPC-9 provincias, IPC-CENDA/IPC-7,
   "Buenos Aires City") encuentran *menciones* a estos índices alternativos del
   período intervenido, pero ninguna con un archivo mensual descargable de
   forma directa en el tiempo de esta búsqueda. Un observatorio de la UNR que
   parecía relevante (`observatorio.unr.edu.ar/indices-de-precios-alternativos-2007-2016`)
   devolvió **403 Forbidden**.
5. Búsqueda específica del lado de INDEC (empalme oficial 2007-2015 para CER):
   **no existe** -- INDEC decidió explícitamente *no* publicar un empalme
   oficial de ese tramo, por el riesgo legal de litigios de bonistas si el
   Estado admitía la magnitud real de la manipulación.

**No se sustituye Bevacqua por otro índice en silencio.** Lo que sigue (3-4)
es lo que sí se pudo hacer sin esa serie: el chequeo de magnitud de los otros
dos ajustes que aplica Tornarolli (ponderadores 2013-2015, no respuesta) --
quedan descartados como causa principal, consistente con D29. **La
recalculación de tasas propias 2003-2015 con el índice de Bevacqua queda
bloqueada hasta conseguir la serie** -- pendiente de que Iván indique una
fuente, o de evaluar un proxy explícitamente etiquetado como tal (ej.
IPC-San Luis) en otra ronda, con el mismo aviso.

### Magnitud de los otros dos ajustes de Tornarolli (ponderadores, no respuesta) -- no explican el gap

**[confirmado, leído directamente del cuerpo del paper vía `pdftotext`, no de
un resumen]** Los dos ajustes que Tornarolli aplica y que nosotros no
replicamos todavía son chicos frente a la brecha de 27-34pp que vimos en
D29:

- **Ponderadores 2013-2015** (Sección 3 del paper): reduce la indigencia
  0,1-0,2pp en los semestres afectados, y la pobreza 0,5-0,8pp en 2014-2015-I.
  Solo aplica a 4 semestres puntuales (2013-II a 2015-I), no a toda la serie.
- **Método de no respuesta** (Sección 4): comparando 2015-I con el puente
  2016-2017, el método propio de Tornarolli (hot-deck) reduce la indigencia
  ~1,3-1,4pp y la pobreza ~3,5-4,5pp frente al método de reponderación que usa
  este repo desde 2016 (D3/D25) -- la magnitud que ya tenía anotada D25 como
  pendiente.

**Combinados, explican como mucho ~5pp de las ~27-34pp de brecha en 2003-II**
-- ni por separado ni juntos alcanzan para explicar el gap observado. Esto
**refuerza, no debilita, el diagnóstico de D29** (el índice de precios --
IPIM vs. IPC -- es la causa dominante): aunque se pudiera replicar
exactamente el ajuste de ponderadores y de no respuesta de Tornarolli sin
tener su índice de precios, la brecha seguiría siendo de magnitud similar a
la observada.

---

## Cierre de esta ronda

**Sección 1 (tamaño de muestra)**: 243.901 hogares históricos con match
confirmado en `t+4` -- 2,52x el universo de entrenamiento actual (160.786
solo 2016+ pre-holdout). 0 pares cruzan la frontera metodológica, por
construcción (`CODUSU` incompatible entre eras, D7) -- la era histórica solo
puede sumar pares *dentro* de sí misma. Es una cifra grande, que justifica
seguir evaluando el backcast.

**Sección 2 (backcast Bevacqua)**: **bloqueada en la fuente de datos** -- no
se encontró la serie mensual de Bevacqua como dataset descargable en esta
ronda (búsqueda documentada arriba, escalada en vez de resuelta con un
sustituto silencioso). `retropolacion_pobreza.py` ya quedó parametrizado por
índice (D30), listo para recibir cualquier serie el día que se consiga una.
El chequeo de los otros dos ajustes de Tornarolli (ponderadores, no
respuesta) descarta que sean la causa principal del gap -- consistente con
D29, no lo contradice.

Ver D30 en el log para el detalle completo. Frena acá -- la decisión de cómo
seguir (buscar la fuente de Bevacqua por otro canal, evaluar un proxy
explícito como IPC-San Luis, o dar por cerrada esta línea y quedarse con
2016+ como núcleo) queda para Iván.

---

## Sección 3 — Isla 2003-2006 con coeficientes PPCC (D37-D39, pedido de Iván)

D31 cerró la retropolación histórica con el backcast FACPCE (sección 1
arriba). Esta sección es un intento **distinto**: no deflacta nada -- usa
valores reales de CBA/CBT publicados por INDEC para GBA (`sh-cba2.xls`,
sept-2000 a dic-2013) y los coeficientes de Paridad de Poder de Compra del
Consumidor (PPCC) que INDEC aplicaba en su "metodología de transición"
(vigente sin cambios documentados desde 2001/2002 hasta el rediseño de
2016 -- confirmado leyendo `EPH_metodologia_22_pobreza.pdf`: la
incorporación de canastas con estructura de consumo propia por región "no
surge de un planteo metodológico novedoso" sino que recién se pudo hacer en
2016, es decir, el mecanismo PPCC siguió vigente sin reemplazo hasta
entonces) para construir líneas **regionales**, no solo de GBA, en
2003T3-2006T4.

**Fuentes nuevas, descargadas y versionadas** en
`data/canastas_indec/retropolacion/indec_pre_intervencion/`:
- `sh-cba2.xls`: INDEC, *Valores de la Canasta Básica de Alimentos (CBA) y
  de la Canasta Básica Total (CBT) para el adulto equivalente, desde
  septiembre 2000 en adelante* (Gran Buenos Aires).
- `ppcc_regional_abril_2001.csv`: transcripción de INDEC, *Incidencia de la
  pobreza en los aglomerados urbanos, mayo de 2001* (19/2/2002), tabla de
  CBA/CBT por región a abril de 2001 -- verificado con `pdftotext -layout`
  contra `pob_tmay01.pdf`, no de un resumen automático (mismo criterio que
  D25). Coincide, al punto decimal, con la fila "Alimentos y bebidas"/"Nivel
  general" de `PPCC_2001_Regiones_Dos_digitos.pdf` (INDEC, *Paridades de
  Poder de Compra del Consumidor en abril de 2001*) -- ambos documentos
  describen el mismo mecanismo.
- `sh_pobrezaeindigencia_continua.xls`: serie histórica de tasas de
  pobreza/indigencia por región estadística desde 1S2003 (citada en
  `pob_tot_2sem12.pdf`, ya en el repo) -- permite validar por región, no
  solo nacional.

**Código nuevo** (`src/retropolacion_pobreza.py`, con tests):
`descargar_xls_indec`, `cargar_cba_cbt_gba_historico`, `cargar_ppcc_regional`,
`construir_canasta_regional_ppcc`, `cargar_tasas_regionales_historico`.

In [ ]:
PRE_INTERVENCION = CANASTAS / "retropolacion" / "indec_pre_intervencion"

manifest_cba_cbt = retropolacion_pobreza.descargar_xls_indec(retropolacion_pobreza.URL_CBA_CBT_GBA_HISTORICO, PRE_INTERVENCION / "sh-cba2.xls")
manifest_tasas = retropolacion_pobreza.descargar_xls_indec(retropolacion_pobreza.URL_POBREZA_REGIONAL_HISTORICO,
                                         PRE_INTERVENCION / "sh_pobrezaeindigencia_continua.xls")
print(f"sh-cba2.xls: {len(manifest_cba_cbt):,} bytes")
print(f"sh_pobrezaeindigencia_continua.xls: {len(manifest_tasas):,} bytes")


### Canasta regional PPCC, 2003T3-2006T4

`construir_canasta_regional_ppcc` multiplica el valor real de GBA de cada
mes por el ratio región/GBA de abril de 2001 (`ppcc_regional_abril_2001.csv`)
-- mismo `canasta_regional.agregar_trimestral("a_mismo_trimestre")` que el
resto del repo (D26) para pasar a trimestral.

In [ ]:
cba_cbt_gba_historico = retropolacion_pobreza.cargar_cba_cbt_gba_historico(PRE_INTERVENCION / "sh-cba2.xls")
ppcc_regional = retropolacion_pobreza.cargar_ppcc_regional(PRE_INTERVENCION / "ppcc_regional_abril_2001.csv")

canasta_ppcc_larga = retropolacion_pobreza.construir_canasta_regional_ppcc(cba_cbt_gba_historico, ppcc_regional, (2003, 7), (2006, 12))
canasta_ppcc_cba = canasta_regional.agregar_trimestral(
    canasta_ppcc_larga[["REGION", "ANO4", "MES", "cba_ae"]], "cba_ae", "a_mismo_trimestre")
canasta_ppcc_cbt = canasta_regional.agregar_trimestral(
    canasta_ppcc_larga[["REGION", "ANO4", "MES", "cbt_ae"]], "cbt_ae", "a_mismo_trimestre")
canasta_ppcc_trimestral = canasta_ppcc_cba.merge(
    canasta_ppcc_cbt, on=["ANO4", "TRIMESTRE", "REGION"])[["ANO4", "TRIMESTRE", "REGION", "cba_ae", "cbt_ae"]]
print(f"{len(canasta_ppcc_trimestral)} filas región x trimestre, "
      f"{canasta_ppcc_cba['canasta_ventana_parcial'].sum()} con ventana parcial")


### Validación: nacional y por región, 7 semestres 2S2003-2S2006

`1S2003` queda afuera (EPH puntual, no continua -- confirmado por Iván).
Ponderación histórica: `PONDERA` para hogares, `PONDERA × n_miembros` para
personas (D28/D30, no existe `PONDIH` antes de 2016). Mismo umbral D25
(≤0,3pp aceptable / 0,3-1pp revisar / >1pp frenar) contra la tasa oficial
por región (`sh_pobrezaeindigencia_continua.xls`).

In [ ]:
COLUMNAS_HOGAR_HIST = ["CODUSU", "NRO_HOGAR", "AGLOMERADO", "ITF", "ingreso_no_declarado"]
COLUMNAS_IND_HIST = ["CODUSU", "NRO_HOGAR", "CH04", "CH06", "PONDERA"]
SEMESTRES_VALIDACION = [(2003, 2), (2004, 1), (2004, 2), (2005, 1), (2005, 2), (2006, 1), (2006, 2)]
ETIQUETA_SEMESTRE = {
    (2003, 2): "Segundo semestre de 2003", (2004, 1): "Primer semestre de 2004",
    (2004, 2): "Segundo semestre de 2004", (2005, 1): "Primer semestre de 2005",
    (2005, 2): "Segundo semestre de 2005", (2006, 1): "Primer semestre de 2006",
    (2006, 2): "Segundo semestre de 2006",
}


def cargar_trimestre_historico(anio: int, trimestre: int) -> pd.DataFrame:
    hogar = pd.read_parquet(NUCLEO / "hogar" / f"ANO4={anio}" / f"TRIMESTRE={trimestre}" / "data.parquet",
                             columns=COLUMNAS_HOGAR_HIST)
    region = pd.read_parquet(ARMONIZADO / "hogar" / f"ANO4={anio}" / f"TRIMESTRE={trimestre}" / "data.parquet",
                              columns=["CODUSU", "NRO_HOGAR", "REGION"])
    region["REGION"] = pd.to_numeric(region["REGION"], errors="coerce").astype("Int8")
    hogar = hogar.merge(region, on=["CODUSU", "NRO_HOGAR"], how="left")
    individual = pd.read_parquet(NUCLEO / "individual" / f"ANO4={anio}" / f"TRIMESTRE={trimestre}" / "data.parquet",
                                  columns=COLUMNAS_IND_HIST)
    canasta_t = canasta_ppcc_trimestral[(canasta_ppcc_trimestral["ANO4"] == anio)
                                         & (canasta_ppcc_trimestral["TRIMESTRE"] == trimestre)]
    distancia = linea_pobreza.calcular_distancia_lp(hogar, individual, canasta_t[["REGION", "cba_ae", "cbt_ae"]])
    pondera_hogar = retropolacion_pobreza.peso_hogar_historico(individual).rename("PONDERA")
    n_miembros = individual.groupby(["CODUSU", "NRO_HOGAR"]).size().rename("n_miembros")
    distancia = distancia.merge(pondera_hogar, on=["CODUSU", "NRO_HOGAR"], how="left")
    distancia = distancia.merge(n_miembros, on=["CODUSU", "NRO_HOGAR"], how="left")
    return distancia


def tasa_ponderada_hist(df: pd.DataFrame, columna: str, peso: pd.Series) -> float:
    valido = df[columna].notna() & peso.notna()
    return 100 * (df.loc[valido, columna].astype(float) * peso[valido]).sum() / peso[valido].sum()


def clasificar_desvio_d25(desvio: float) -> str:
    if abs(desvio) <= 0.3:
        return "aceptable"
    if abs(desvio) <= 1.0:
        return "revisar"
    return "frenar"


tasas_oficiales_regionales = retropolacion_pobreza.cargar_tasas_regionales_historico(PRE_INTERVENCION / "sh_pobrezaeindigencia_continua.xls")
tasas_oficiales_nacionales = pd.read_csv(
    CANASTAS / "retropolacion" / "indec_metodologia_anterior" / "pobreza_1s2003_2s2012.csv")

filas_validacion_ppcc = []
for anio, sem in SEMESTRES_VALIDACION:
    trimestres = (1, 2) if sem == 1 else (3, 4)
    df_semestre = pd.concat([cargar_trimestre_historico(anio, t) for t in trimestres], ignore_index=True)
    df_semestre["peso_personas"] = df_semestre["PONDERA"] * df_semestre["n_miembros"]

    calc_nacional = {
        "pobreza_hogares": tasa_ponderada_hist(df_semestre, "pobre", df_semestre["PONDERA"]),
        "pobreza_personas": tasa_ponderada_hist(df_semestre, "pobre", df_semestre["peso_personas"]),
        "indigencia_hogares": tasa_ponderada_hist(df_semestre, "indigente", df_semestre["PONDERA"]),
        "indigencia_personas": tasa_ponderada_hist(df_semestre, "indigente", df_semestre["peso_personas"]),
    }
    oficial_nacional = tasas_oficiales_nacionales[tasas_oficiales_nacionales["periodo"] == f"{sem}S{anio}"].iloc[0]
    for indicador, valor in calc_nacional.items():
        desvio = valor - oficial_nacional[indicador]
        filas_validacion_ppcc.append({"semestre": f"{sem}S{anio}", "REGION": "nacional", "indicador": indicador,
                                       "calculado": round(valor, 2), "oficial": oficial_nacional[indicador],
                                       "desvio_pp": round(desvio, 2), "clasificacion": clasificar_desvio_d25(desvio)})

    for region, grupo in df_semestre.groupby("REGION"):
        calc_region = {
            "pobreza_hogares": tasa_ponderada_hist(grupo, "pobre", grupo["PONDERA"]),
            "pobreza_personas": tasa_ponderada_hist(grupo, "pobre", grupo["peso_personas"]),
            "indigencia_hogares": tasa_ponderada_hist(grupo, "indigente", grupo["PONDERA"]),
            "indigencia_personas": tasa_ponderada_hist(grupo, "indigente", grupo["peso_personas"]),
        }
        oficial_region = tasas_oficiales_regionales[
            (tasas_oficiales_regionales["REGION"] == region)
            & (tasas_oficiales_regionales["semestre_label"] == ETIQUETA_SEMESTRE[(anio, sem)])]
        for indicador, valor in calc_region.items():
            fila_oficial = oficial_region[oficial_region["indicador"] == indicador]
            if not len(fila_oficial):
                continue
            oficial = fila_oficial["valor"].iloc[0]
            desvio = valor - oficial
            filas_validacion_ppcc.append({"semestre": f"{sem}S{anio}", "REGION": region, "indicador": indicador,
                                           "calculado": round(valor, 2), "oficial": oficial,
                                           "desvio_pp": round(desvio, 2), "clasificacion": clasificar_desvio_d25(desvio)})

validacion_ppcc = pd.DataFrame(filas_validacion_ppcc)
validacion_ppcc.to_csv(META / "retropolacion_ppcc_validacion_2003_2006.csv", index=False)
print(f"{len(validacion_ppcc)} filas -- data/meta/retropolacion_ppcc_validacion_2003_2006.csv")
print("clasificación nacional:")
display(validacion_ppcc[validacion_ppcc["REGION"] == "nacional"]["clasificacion"].value_counts())
print("clasificación por región (incluye GBA sola):")
display(validacion_ppcc[validacion_ppcc["REGION"] != "nacional"].groupby("REGION")["clasificacion"].value_counts())


### Conclusión de la Sección 3 -- no se escribe la isla esta ronda (D39)

Ver D39 en `docs/decisiones_metodologicas.md` para el detalle completo y el
número exacto. Resumen: ni la línea PPCC regional ni GBA sola (dentro de la
misma corrida) pasan el umbral ≤1pp que usa el resto del repo (D25) de forma
limpia -- el gate que acordamos con Iván ("si no valida con PPCC, probar
GBA sola; si tampoco, no se escribe nada") da **no se escribe nada**. Mucho
mejor que el backcast FACPCE (D28-D30, desvíos de 27-40pp) pero no limpio.
Hallazgo pendiente de seguir si Iván quiere invertir más: la tabla de
adulto equivalente vigente en este repo es la de 2020 (post-rediseño
2016) -- `EPH_metodologia_22_pobreza.pdf` confirma que la tabla de
equivalencias cambió recién en 2016, así que la tabla 2003-2006 real era
distinta y no está localizada todavía; podría explicar parte del desvío
residual.